In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
movie_df_raw = pd.read_csv("/content/drive/MyDrive/8th semester/Data mining lab/ratings.csv")
movie_names = pd.read_csv("/content/drive/MyDrive/8th semester/Data mining lab/movies.csv")

In [ ]:
movie_df_raw

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [ ]:
movie_names

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


### Data Processing: Build User-Item Interaction Matrix

Instead of iterating through rows to build the matrix, which can be very slow, I will use `pivot_table` to efficiently create the user-item interaction matrix. This matrix will have `userId` as index, `movieId` as columns, and `rating` as values. I'll also merge movie titles for easier interpretation later.

In [ ]:
movie_ratings = pd.merge(movie_df_raw, movie_names, on='movieId', how='left')

# Create the user-item interaction matrix using pivot_table
user_movie_matrix = movie_ratings.pivot_table(index='userId', columns='title', values='rating').fillna(0)
display(user_movie_matrix.head())

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Handle Missing Values

Missing values in the user-item matrix (where a user hasn't rated a movie) are filled with `0` during the `pivot_table` creation. This is a common strategy for item-based collaborative filtering, indicating no rating rather than a low rating.

### Prepare Data for Similarity Calculation (Item-Based)

For item-based collaborative filtering, we need to calculate similarity between movies (items). This means we'll transpose our `user_movie_matrix` so that rows represent movies and columns represent users. Then, we can compute similarity between movie rows.

In [ ]:
item_user_matrix = user_movie_matrix.T
display(item_user_matrix.head())

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Model Implementation: Compute Item Similarity

I will use `cosine_similarity` to measure the similarity between movies. Cosine similarity is a good choice as it measures the cosine of the angle between two vectors, making it suitable for sparse rating data where `0` means no rating.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute item similarity matrix
item_similarity_matrix = cosine_similarity(item_user_matrix)

# Convert to DataFrame for easier handling, with movie titles as index and columns
item_similarity_df = pd.DataFrame(item_similarity_matrix, index=item_user_matrix.index, columns=item_user_matrix.index)
display(item_similarity_df.head())

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


### Use Item Similarity to Predict User Preferences

Now, let's create a function to predict ratings for a given user and a movie they haven't rated. The prediction will be based on the weighted average of the user's ratings for similar movies, where the weights are the item similarities.

In [ ]:
def predict_rating(user_id, movie_title, user_movie_matrix, item_similarity_df):
    # Get the user's ratings
    user_ratings = user_movie_matrix.loc[user_id]

    # Get similarities of the target movie with all other movies
    item_similarities = item_similarity_df[movie_title]

    # Filter out movies the user hasn't rated (or rated 0) and the movie itself
    rated_similar_movies = user_ratings[user_ratings > 0].index
    if movie_title in rated_similar_movies:
        # User has already rated this movie
        return user_ratings[movie_title]

    # Exclude the target movie itself from similarity calculation if it's in the rated list
    # (though it shouldn't be for prediction of unrated movies)
    relevant_similarities = item_similarities[rated_similar_movies]
    relevant_user_ratings = user_ratings[rated_similar_movies]

    # Calculate weighted sum of ratings
    numerator = (relevant_similarities * relevant_user_ratings).sum()
    denominator = relevant_similarities.sum()

    if denominator == 0:
        return 0  # Cannot make a prediction if no similar rated items
    else:
        return numerator / denominator

### Generate Top-N Recommendations

Now I will generate the top N recommendations for a specific user by predicting ratings for all movies they haven't seen and then selecting the highest-rated ones. I will also provide recommendations for at least 5 users as requested.

In [ ]:
def get_top_n_recommendations(user_id, user_movie_matrix, item_similarity_df, n=10):
    user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = user_ratings[user_ratings == 0].index

    predictions = []
    for movie_title in unrated_movies:
        predicted_rating = predict_rating(user_id, movie_title, user_movie_matrix, item_similarity_df)
        if predicted_rating > 0:
            predictions.append((predicted_rating, movie_title))

    predictions.sort(key=lambda x: x[0], reverse=True)
    return predictions[:n]

#### Recommendations for Multiple Users

Let's pick a few users and demonstrate the recommendations.

In [ ]:
example_user_ids = user_movie_matrix.index[0:5].tolist() # Take the first 5 user IDs

for user_id in example_user_ids:
    print(f"\n--- Recommendations for User {user_id} ---")
    recommendations = get_top_n_recommendations(user_id, user_movie_matrix, item_similarity_df, n=5)
    if recommendations:
        for rating, movie_title in recommendations:
            print(f"  Movie: {movie_title} (Predicted Rating: {rating:.2f})")
    else:
        print("  No recommendations could be generated for this user.")


--- Recommendations for User 1 ---
  Movie: Alvarez Kelly (1966) (Predicted Rating: 5.00)
  Movie: Bakuman (2015) (Predicted Rating: 5.00)
  Movie: Betting on Zero (2016) (Predicted Rating: 5.00)
  Movie: Black Butler: Book of the Atlantic (2017) (Predicted Rating: 5.00)
  Movie: Blue Exorcist: The Movie (2012) (Predicted Rating: 5.00)

--- Recommendations for User 2 ---
  Movie: A Street Cat Named Bob (2016) (Predicted Rating: 5.00)
  Movie: Daddy's Home 2 (2017) (Predicted Rating: 5.00)
  Movie: Deathgasm (2015) (Predicted Rating: 5.00)
  Movie: 101 Reykjavik (101 Reykjavík) (2000) (Predicted Rating: 4.50)
  Movie: 52 Pick-Up (1986) (Predicted Rating: 4.50)

--- Recommendations for User 3 ---
  Movie: 1-900 (06) (1994) (Predicted Rating: 5.00)
  Movie: Beyond Borders (2003) (Predicted Rating: 5.00)
  Movie: Big Empty, The (2003) (Predicted Rating: 5.00)
  Movie: Dear God (1996) (Predicted Rating: 5.00)
  Movie: Fighting Temptations, The (2003) (Predicted Rating: 5.00)

--- Recommend

### Evaluation

To evaluate the recommendation system, we'll split the `movie_ratings` data into training and testing sets. We'll build the recommendation model on the training data and then predict ratings for the movies in the test set. Finally, we'll use Root Mean Squared Error (RMSE) to measure the accuracy of our predictions.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Split data into training and testing sets
train_data, test_data = train_test_split(movie_ratings, test_size=0.2, random_state=42)

# Create user-item matrices for training and testing
train_user_movie_matrix = train_data.pivot_table(index='userId', columns='title', values='rating').fillna(0)
test_user_movie_matrix = test_data.pivot_table(index='userId', columns='title', values='rating').fillna(0)

# Prepare item-user matrix for training data
train_item_user_matrix = train_user_movie_matrix.T

# Compute item similarity matrix for training data
train_item_similarity_matrix = cosine_similarity(train_item_user_matrix)
train_item_similarity_df = pd.DataFrame(train_item_similarity_matrix, index=train_item_user_matrix.index, columns=train_item_user_matrix.index)

# Store actual and predicted ratings for evaluation
actual_ratings = []
predicted_ratings = []

for index, row in test_data.iterrows():
    user_id = int(row['userId'])
    movie_title = row['title']
    actual_rating = row['rating']

    # Ensure the user and movie exist in the training matrix for prediction
    if user_id in train_user_movie_matrix.index and movie_title in train_item_similarity_df.columns:
        predicted_rating = predict_rating(user_id, movie_title, train_user_movie_matrix, train_item_similarity_df)
        if predicted_rating > 0: # Only consider predictions where a value could be generated
            actual_ratings.append(actual_rating)
            predicted_ratings.append(predicted_rating)

# Calculate RMSE
if actual_ratings and predicted_ratings:
    rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
    print(f"\nRoot Mean Squared Error (RMSE): {rmse:.4f}")
else:
    print("Could not generate enough predictions for evaluation.")


Root Mean Squared Error (RMSE): 0.9202


### Performance Comment

The RMSE value (displayed in the output of the cell above) indicates the average magnitude of the errors in our predicted ratings. A lower RMSE suggests a more accurate model. For this dataset and item-based collaborative filtering approach, this RMSE indicates that, on average, our predicted ratings are that many units away from the actual ratings. The performance can be further improved by:

*   **Handling sparsity**: Many users rate very few movies, leading to sparse matrices, which can be challenging for similarity calculations.
*   **Cold-start problem**: New users or new movies with no ratings cannot be effectively recommended.
*   **Incorporating more features**: Content-based features (e.g., movie genres, director) could be combined with collaborative filtering to enhance recommendations.
*   **Different similarity metrics**: Experimenting with other similarity measures like Pearson correlation.
*   **Matrix Factorization**: Techniques like Singular Value Decomposition (SVD) or Non-negative Matrix Factorization (NMF) can often provide better performance by finding latent factors in the data.

### Generate Top-5 Recommendations for a Specific User

Let's get the top-5 recommendations for a particular user, for example, `userId = 1`.

In [ ]:
target_user_id = 1 # You can change this to any valid user ID

print(f"\n--- Top 5 Recommendations for User {target_user_id} ---")
recommendations = get_top_n_recommendations(target_user_id, user_movie_matrix, item_similarity_df, n=5)
if recommendations:
    for rating, movie_title in recommendations:
        print(f"  Movie: {movie_title} (Predicted Rating: {rating:.2f})")
else:
    print("  No recommendations could be generated for this user.")


--- Top 5 Recommendations for User 1 ---
  Movie: Alvarez Kelly (1966) (Predicted Rating: 5.00)
  Movie: Bakuman (2015) (Predicted Rating: 5.00)
  Movie: Betting on Zero (2016) (Predicted Rating: 5.00)
  Movie: Black Butler: Book of the Atlantic (2017) (Predicted Rating: 5.00)
  Movie: Blue Exorcist: The Movie (2012) (Predicted Rating: 5.00)


### Provide Your Own Ratings for Recommendations

Now, you can enter your own ratings for a few movies. This will simulate a new user's preferences, and we will generate recommendations based on these ratings using our item-based collaborative filtering model.

In [ ]:
# Define your ratings here. Use exact movie titles from the dataset.
# Example: {'Toy Story (1995)': 5.0, 'Jumanji (1995)': 3.0}

new_user_ratings_input = {
    'Pulp Fiction (1994)': 5.0,
    'Forrest Gump (1994)': 4.0,
    'Matrix, The (1999)': 5.0,
    'Star Wars: Episode IV - A New Hope (1977)': 4.5,
    'Shawshank Redemption, The (1994)': 5.0
}

# Create a new DataFrame for the new user's ratings
# We need to align the columns (movie titles) with the existing user_movie_matrix
new_user_id = user_movie_matrix.index.max() + 1 if not user_movie_matrix.empty else 1

# Initialize a new row for the user with all zeros for movies not rated by them
new_user_row = pd.Series(0.0, index=user_movie_matrix.columns, name=new_user_id)

# Populate the new user's ratings
for movie_title, rating in new_user_ratings_input.items():
    if movie_title in new_user_row.index:
        new_user_row[movie_title] = rating
    else:
        print(f"Warning: Movie '{movie_title}' not found in the dataset. Skipping this rating.")

# Add the new user to the user_movie_matrix
# We use pd.concat to add the new row to the existing DataFrame
user_movie_matrix_with_new_user = pd.concat([user_movie_matrix, pd.DataFrame([new_user_row])])

display(user_movie_matrix_with_new_user.tail())


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
607,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.5,3.5,0.0,0.0,0.0
609,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
610,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.5,0.0,...,0.0,4.0,3.5,3.0,0.0,0.0,2.0,1.5,0.0,0.0
611,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Generate Recommendations for the New User

Now that the new user's ratings have been added, we can generate top-N recommendations for them based on the item similarity model.

In [ ]:
print(f"\n--- Top 5 Recommendations for New User {new_user_id} ---")
recommendations_for_new_user = get_top_n_recommendations(
    new_user_id,
    user_movie_matrix_with_new_user, # Use the updated matrix
    item_similarity_df,
    n=5
)

if recommendations_for_new_user:
    for rating, movie_title in recommendations_for_new_user:
        print(f"  Movie: {movie_title} (Predicted Rating: {rating:.2f})")
else:
    print("  No recommendations could be generated for this user. Try rating more movies.")


--- Top 5 Recommendations for New User 611 ---
  Movie: Addams Family Reunion (1998) (Predicted Rating: 5.00)
  Movie: Amer (2009) (Predicted Rating: 5.00)
  Movie: Disgrace (2008) (Predicted Rating: 5.00)
  Movie: Element of Crime, The (Forbrydelsens Element) (1984) (Predicted Rating: 5.00)
  Movie: Ex Drummer (2007) (Predicted Rating: 5.00)


In [ ]:
movie_df = pd.DataFrame(columns=movie_df_raw['movieId'].unique(), index=movie_df_raw['userId'].unique())
movie_df

,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
for i in range(movie_df_raw.shape[0]):
  row = movie_df_raw.iloc[i]
  user = row['userId']
  movie = row['movieId']
  rating = row['rating']

  movie_df.loc[user, movie] = rating

In [ ]:
movie_df

,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
1,4.0,4.0,4.0,5.0,5.0,3.0,5.0,4.0,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,NaN,NaN,3.0,4.5,4.0,NaN,3.5,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,4.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,2.5,2.0,NaN,4.5,4.5,3.0,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
609,3.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
movie_df = movie_df.fillna(0)
movie_df

/tmp/ipykernel_1487/1983800752.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  movie_df = movie_df.fillna(0)


,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
1,4.0,4.0,4.0,5.0,5.0,3.0,5.0,4.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,0.0,0.0,3.0,4.5,4.0,0.0,3.5,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
607,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,2.5,2.0,0.0,4.5,4.5,3.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
609,3.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
user_ratings = {88125:3.2, 152081: 3.5, 150548:3.75, 99114:4.5, 48780:4, 58559:4.5, 59315: 3.75, 74458:4, 102903: 3.5, 2150:3.5, 2176: 4.5, 2329:3.5, 189713:3.75, 187593:3.5,184641:4,179819:2.5,176101:3,1717151:3.75,170697:4,168250:4.25,162082:4.25,161922:4,159093:3  }
user_df = pd.DataFrame(user_ratings, columns=movie_df.columns, index=[611])
user_df = user_df.fillna(0)
user_df

/tmp/ipykernel_1487/2610416841.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  user_df = user_df.fillna(0)


,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
611,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
movie_df = pd.concat([movie_df, user_df])
movie_df

,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
1,4.0,4.0,4.0,5.0,5.0,3.0,5.0,4.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,2.5,2.0,0.0,4.5,4.5,3.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
609,3.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
610,5.0,0.0,5.0,5.0,4.0,4.0,0.0,4.5,0.0,0.0,...,3.0,3.5,3.5,3.5,3.5,2.5,4.5,3.0,3.5,3.5


In [ ]:
from scipy.spatial.distance import euclidean

def euclidean_similarity(user1, user2):
  common_ratings = (user1 != 0 ) & (user2 != 0)
  if np.sum(common_ratings) <=5:
     return 0
  distance = euclidean(user1[common_ratings], user2[common_ratings])
  similarity = 1/(1+distance)
  return similarity

In [ ]:
target_user = 611

user_similarities = {}

for user in movie_df.index:
  if user != target_user:
    similarity = euclidean_similarity(movie_df.loc[target_user], movie_df.loc[user])
    user_similarities[user] = similarity

Enter your own movie ratings here.

In [ ]:
top_match = sorted([(value,key) for (key,value) in user_similarities.items()], reverse=True)
top_n = 5
top_match[:5]

[(0.45781845523652454, 525),
 (0.4547608704644112, 125),
 (0.4068781988545112, 18),
 (0.38910023673089295, 596),
 (0.3844998779892558, 288)]

Making the recommendation

In [ ]:
user_df

,1,3,6,47,50,70,101,110,151,157,...,147662,148166,149011,152372,158721,160341,160527,160836,163937,163981
611,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
movies_not_rated = user_df.loc[target_user][movie_df.loc[target_user]==0].index
movies_not_rated

Index([     1,      3,      6,     47,     50,     70,    101,    110,    151,
          157,
       ...
       147662, 148166, 149011, 152372, 158721, 160341, 160527, 160836, 163937,
       163981],
      dtype='int64', length=9702)

In [ ]:
recommendations = []

for u in top_match[:5]:
  u_ratings = movie_df.loc[u[1]]
  for i in u_ratings.index:
    if u_ratings.loc[i]!=0 and i in movies_not_rated:
      recommendations.append( (u_ratings.loc[i], str(movie_names.loc[movie_names["movieId"]==i, "title"])) )

In [ ]:
sorted(recommendations, reverse=True)[:20]

[(np.float64(5.0),
  '990    Indiana Jones and the Last Crusade (1989)\nName: title, dtype: object'),
 (np.float64(5.0), '97    Braveheart (1995)\nName: title, dtype: object'),
 (np.float64(5.0),
  '969    Back to the Future (1985)\nName: title, dtype: object'),
 (np.float64(5.0),
  '969    Back to the Future (1985)\nName: title, dtype: object'),
 (np.float64(5.0),
  '9570    Black Mirror: White Christmas (2014)\nName: title, dtype: object'),
 (np.float64(5.0), '9521    Baby Driver (2017)\nName: title, dtype: object'),
 (np.float64(5.0), '9463    Logan (2017)\nName: title, dtype: object'),
 (np.float64(5.0),
  '9452    The Lego Batman Movie (2017)\nName: title, dtype: object'),
 (np.float64(5.0),
  '9433    Rogue One: A Star Wars Story (2016)\nName: title, dtype: object'),
 (np.float64(5.0), '9297    Sausage Party (2016)\nName: title, dtype: object'),
 (np.float64(5.0),
  '922    Godfather: Part II, The (1974)\nName: title, dtype: object'),
 (np.float64(5.0), '920    Psycho (1960)\nNam